# RMSProp
AdaGrad against RMSProp on the students data, the RMSProp average unrolled,
and AdaGrad against RMSProp on MNIST over 30 epochs.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"      # hide TensorFlow's start-up messages
import numpy as np
import pandas as pd
import keras
import tensorflow as tf

tf.config.experimental.enable_op_determinism()   # same numbers on every run
np.set_printoptions(precision=4, suppress=True)

## The students data (IIT sparse, package target), from (m, b) = (-4, -4), learning rate 0.2 for both

In [ ]:
d = pd.read_csv("data/students.csv")
X = np.c_[d.iit, np.ones(len(d))]
y = d.package.to_numpy()
best = np.linalg.lstsq(X, y, rcond=None)[0]
loss = lambda p: np.mean((y - X @ p) ** 2)
grad = lambda p: -2 * X.T @ (y - X @ p) / len(y)

def run(kind, eta=0.2, steps=300, beta=0.9):
    """AdaGrad: v = v + g^2.  RMSProp: v = beta v + (1 - beta) g^2.  Then p = p - eta g / sqrt(v)."""
    p, v = np.array([-4.0, -4.0]), np.zeros(2)
    path, vs = [p.copy()], []
    for _ in range(steps):
        g = grad(p)
        v = v + g ** 2 if kind == "adagrad" else beta * v + (1 - beta) * g ** 2
        p = p - eta * g / (np.sqrt(v) + 1e-8)
        path.append(p.copy())
        vs.append(v.copy())
    return np.array(path), np.array(vs)

for kind in ("adagrad", "rmsprop"):
    P, Vs = run(kind)
    gap = np.array([loss(p) for p in P]) - loss(best)
    reached = int(np.argmax(gap < 0.01)) if (gap < 0.01).any() else None
    print(f"{kind}: loss within 0.01 of the best after {reached} steps; (m, b) after 100 steps {P[100].round(2)}, "
          f"after 300 {P[-1].round(2)}; best {best.round(2)}")
    print(f"   v of b at steps 1, 10, 300: {Vs[[0, 9, 299], 1].round(3)}; "
          f"eta/sqrt(v) of b: {(0.2 / np.sqrt(Vs[[0, 9, 299], 1])).round(4)}")
P, _ = run("rmsprop")
print("RMSProp, last 100 steps: largest distance from the best", np.abs(P[-100:] - best).max().round(3))

## The EWMA of squared gradients, unrolled: v3 = (1 - beta)(beta^2 g1^2 + beta g2^2 + g3^2)

In [ ]:
beta, g = 0.9, np.array([3.0, 2.0, 1.0])
v = 0.0
for gi in g:
    v = beta * v + (1 - beta) * gi ** 2
print("loop:", round(v, 4), " unrolled:", round((1 - beta) * (beta**2 * 9 + beta * 4 + 1), 4),
      " AdaGrad's sum:", (g ** 2).sum())

## MNIST: AdaGrad against RMSProp, 30 epochs
10,000 training images, the 10,000 test images for validation, batch size 64, learning rate 0.001 for both
(Keras' default for both), epsilon 1e-7, both accumulators start at 0. Three seeds each.
After every epoch we record the median effective learning rate eta / sqrt(v) over all weights.

In [ ]:
(x, yy), (x_test, y_test) = keras.datasets.mnist.load_data()
x = x[:10000].reshape(-1, 784).astype("float32") / 255
yy = yy[:10000]
x_test = x_test.reshape(-1, 784).astype("float32") / 255

class EffectiveRate(keras.callbacks.Callback):
    """Median of eta / sqrt(v + eps) over all weights, after each epoch."""
    def __init__(self, opt, name):
        super().__init__()
        self.opt, self.name, self.values = opt, name, []
    def on_epoch_end(self, epoch, logs=None):
        acc = [v.numpy().ravel() for v in self.opt.variables if self.name in v.path]
        acc = np.concatenate(acc)
        self.values.append(float(np.median(float(self.opt.learning_rate.numpy()) / np.sqrt(acc + 1e-7))))

def train(kind, seed, eta=0.001, epochs=30):
    keras.utils.set_random_seed(seed)
    model = keras.Sequential([keras.Input(shape=(784,)),
                              keras.layers.Dense(128, activation="relu"),
                              keras.layers.Dense(64, activation="relu"),
                              keras.layers.Dense(10, activation="softmax")])
    if kind == "adagrad":
        opt, var = keras.optimizers.Adagrad(learning_rate=eta, initial_accumulator_value=0.0), "accumulator"
    else:
        opt, var = keras.optimizers.RMSprop(learning_rate=eta, rho=0.9), "velocity"
    model.compile(optimizer=opt, loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    rate = EffectiveRate(opt, var)
    h = model.fit(x, yy, epochs=epochs, batch_size=64, validation_data=(x_test, y_test), verbose=0,
                  callbacks=[rate]).history
    return h, rate.values

rows = []
for kind in ("adagrad", "rmsprop"):
    for seed in range(3):
        h, rates = train(kind, seed)
        for e in range(30):
            rows.append(dict(optimizer=kind, seed=seed, epoch=e + 1, loss=h["loss"][e],
                             val_accuracy=h["val_accuracy"][e], eff_lr=rates[e]))
        print(f"{kind} seed {seed}: loss epoch 30 {h['loss'][-1]:.4f}, val accuracy {h['val_accuracy'][-1]:.4f}")
runs = pd.DataFrame(rows)
runs.to_csv("data/mnist_rmsprop.csv", index=False)
mean = runs.groupby(["optimizer", "epoch"])[["loss", "val_accuracy", "eff_lr"]].mean()
for kind in ("adagrad", "rmsprop"):
    m = mean.loc[kind]
    print(f"{kind}: loss epoch 1/10/30 {m.loss[1]:.4f} {m.loss[10]:.4f} {m.loss[30]:.4f}; "
          f"val accuracy epoch 30 {m.val_accuracy[30]:.4f}; median effective rate epoch 1/10/30 "
          f"{m.eff_lr[1]:.4f} {m.eff_lr[10]:.4f} {m.eff_lr[30]:.4f}")

## AdaGrad with larger learning rates (seed 0)

In [ ]:
for eta in (0.01, 0.03, 0.1):
    h, rates = train("adagrad", 0, eta=eta)
    print(f"AdaGrad eta {eta}: loss epoch 10 {h['loss'][9]:.4f}, epoch 30 {h['loss'][-1]:.4f}, "
          f"val accuracy {h['val_accuracy'][-1]:.4f}")